# Project 1: Nigerian Retail Sales — Exploratory Data Analysis with the Tidyverse

**Prodigy Training Hub | R Programming Masterclass Series | In Partnership with Zedcrest Capital**

---

## Project Overview

This project introduces Exploratory Data Analysis (EDA) using R and the tidyverse ecosystem. The business scenario is a fictional Nigerian retail company — **NigeriaRetail Co.** — which operates 40 stores across 10 states and six product categories. The dataset contains **1,500 synthetic transactions** from the 2023 financial year, covering states across all six geopolitical regions of Nigeria.

EDA is the critical first phase of any data project. Before building models or drawing business conclusions, every analyst must understand the structure, quality, and distribution of their data. Skipping EDA leads to biased models, incorrect summaries, and misleading recommendations.

---

## Learning Objectives

By the end of this notebook you will be able to:

- Set up an R Markdown / Jupyter environment with the tidyverse and supporting packages
- Generate and inspect a synthetic Nigerian retail dataset using `tibble()` and `sample()`
- Apply the full tidyverse EDA workflow: inspect → clean → transform → aggregate → visualize
- Use the six core `dplyr` verbs fluently: `filter`, `select`, `mutate`, `group_by`, `summarise`, `arrange`
- Reshape data between wide and long formats using `pivot_wider()` and `pivot_longer()`
- Handle missing values with three distinct imputation strategies
- Build 10 publication-quality `ggplot2` charts covering six chart types
- Extract and communicate data-driven business insights from a Nigerian retail context

---

## Packages Used

| Package | Role | Key Functions |
|---------|------|---------------|
| `tidyverse` | Meta-package: dplyr, ggplot2, tidyr, readr, stringr, forcats, purrr, tibble | Core data manipulation and visualization |
| `janitor` | Column name cleaning and frequency tables | `clean_names()`, `tabyl()` |
| `skimr` | Rich statistical summaries | `skim()` |
| `lubridate` | Date parsing and extraction | `month()`, `quarter()`, `week()`, `wday()` |
| `scales` | Number formatting for charts and tables | `comma()`, `percent()` |
| `knitr` | Clean table rendering | `kable()` |

---

## Dataset Schema

| Column | Type | Description |
|--------|------|-------------|
| `transaction_id` | character | Unique ID per transaction (TXN-00001 to TXN-01500) |
| `date` | Date | Transaction date within 2023 |
| `state` | character | Nigerian state (10 states, probability-weighted) |
| `store_id` | character | Store identifier (STR-001 to STR-040) |
| `category` | character | Product category (6 categories) |
| `units_sold` | integer | Units purchased per transaction (1–50) |
| `unit_price_ngn` | numeric | Price per unit in Naira (₦500–₦150,000) |
| `discount_pct` | numeric | Percentage discount applied (0, 5, 10, 15, 20; 2% missing) |
| `payment_method` | character | Payment channel (Cash, POS, Bank Transfer, Mobile Money) |
| `customer_age` | integer | Customer age in years (18–65; 4% missing) |
| `customer_gender` | character | Customer gender (Male, Female; 1% missing) |
| `returns` | integer | Return flag (0 = No, 1 = Yes) |

---


## Cell 1: Load Libraries

### Explanation

In R, packages extend the base language with additional functions. Unlike Python where you import selectively, R's `library()` loads the entire package into your session.

**`tidyverse`** is a meta-package — a collection of eight packages sharing a unified design philosophy. When you call `library(tidyverse)`, R simultaneously loads:
- `dplyr` — grammar of data manipulation (filter, select, mutate, group_by, summarise)
- `ggplot2` — grammar of graphics for visualization
- `tidyr` — reshaping data between wide and long formats
- `readr` — fast, consistent CSV and flat file reading
- `stringr` — string manipulation built around regular expressions
- `purrr` — functional programming (map, reduce, walk)
- `forcats` — tools for working with factors (categorical variables)
- `tibble` — a modern, stricter version of R's data frame

**`janitor`** provides `clean_names()` which standardises column names to snake_case (lowercase, underscores, no spaces) — called immediately after loading any external dataset.

**`skimr`** provides `skim()`, a dramatically richer replacement for `summary()`. It shows missing value counts, inline histograms, quantiles, and string statistics all in one call.

**`lubridate`** makes date extraction intuitive. `month(date, label=TRUE)` returns `Jan`, `Feb`... instead of integers. `wday(date, label=TRUE)` returns `Monday`, `Tuesday`... as ordered factors that ggplot2 sorts correctly on axes.

**`scales`** provides number formatters: `comma(1234567)` returns `"1,234,567"` and `percent(0.456, accuracy=0.1)` returns `"45.6%"`. Essential for human-readable chart labels.

**`knitr`** provides `kable()` which renders data frames as clean HTML tables with captions.


In [1]:
library(tidyverse)   # Core: dplyr, ggplot2, tidyr, readr, stringr, purrr, forcats
library(janitor)     # clean_names(), tabyl()
library(skimr)       # skim() - rich summary statistics
library(lubridate)   # Date parsing and manipulation
library(scales)      # Number formatting (comma, percent)
library(knitr)       # kable() for clean table output

cat("All libraries loaded successfully.\n")

Warning message:
"package 'tidyverse' was built under R version 4.5.3"
Warning message:
"package 'ggplot2' was built under R version 4.5.3"
Warning message:
"package 'tibble' was built under R version 4.5.3"
Warning message:
"package 'tidyr' was built under R version 4.5.3"
Warning message:
"package 'readr' was built under R version 4.5.3"
Warning message:
"package 'purrr' was built under R version 4.5.3"
Warning message:
"package 'dplyr' was built under R version 4.5.3"
Warning message:
"package 'forcats' was built under R version 4.5.3"
Warning message:
"package 'lubridate' was built under R version 4.5.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()

All libraries loaded successfully.


## Cell 2: Create the Nigerian Retail Dataset

### Explanation

Since we do not have a live Nigerian retail CSV, we generate a synthetic dataset that faithfully mimics the structure and patterns of a real retail operation.

**`set.seed(42)`** initialises R's random number generator at a fixed point. Every person running this notebook gets identical 'random' data. The number 42 is arbitrary — any integer works.

**Parallel vectors and lookup tibble:** `states` and `regions` are positionally aligned — `states[1]` (Lagos) pairs with `regions[1]` (South West). Storing them as a `tibble` called `state_region` creates a lookup table we `left_join()` onto the main dataset later using `state` as the key column.

**`tibble()` vs `data.frame()`:** `tibble()` is the tidyverse replacement for `data.frame()`. Key differences: columns are evaluated left-to-right (a later column can reference an earlier one), strings are never silently converted to factors, and it prints only the first 10 rows.

**Weighted sampling:** `sample(states, n, replace=TRUE, prob=c(0.25, 0.15, ...))` draws states with unequal probability. Lagos gets weight 0.25 (appearing in ~25% of rows), consistent with its status as Nigeria's commercial capital. Without `prob`, every state appears with equal 10% probability — unrealistic for Nigerian retail geography.

**`str_pad()`:** Converts integers 1–1500 into zero-padded strings: `str_pad(1:n, 5, pad='0')` produces `00001` through `01500`. Without padding, IDs sort alphabetically as TXN-1, TXN-10, TXN-100 — incorrect ordering.

**`round(runif(n, 500, 150000), -1)`** generates random prices between ₦500 and ₦150,000. The negative digits argument `-1` rounds to the nearest 10 — so prices look like ₦4,350 rather than ₦4,347.82.

**`sample(c(0, 0, 0, 5, 10, 15, 20), n, replace=TRUE)`** — the three zeros make no-discount transactions three times more frequent than any single discounted level, reflecting real-world pricing where most transactions are full-price.

**Injecting missing values:** `ifelse(runif(n) < 0.04, NA, customer_age)` generates a uniform random number per row; if below 0.04 (4% probability), the value becomes `NA`. This mimics real data entry gaps at point of sale.


In [2]:
set.seed(42)
n <- 1500

# Nigerian states with their geopolitical regions (positionally aligned vectors)
states  <- c("Lagos", "Abuja", "Kano", "Rivers", "Oyo",
             "Delta", "Anambra", "Kaduna", "Enugu", "Ondo")

regions <- c("South West", "North Central", "North West", "South South", "South West",
             "South South", "South East", "North West", "South East", "South West")

# Lookup table for state-to-region mapping
state_region <- tibble(state = states, region = regions)

# Product categories and payment channels
categories      <- c("Electronics", "Fashion & Apparel", "Food & Beverages",
                     "Home & Kitchen", "Health & Beauty", "Agriculture & Farming")
payment_methods <- c("Cash", "POS", "Bank Transfer", "Mobile Money")

# Build the main dataset
raw_retail <- tibble(
  transaction_id  = paste0("TXN-", str_pad(1:n, 5, pad = "0")),
  date            = sample(seq(as.Date("2023-01-01"), as.Date("2023-12-31"), by = "day"),
                            n, replace = TRUE),
  state           = sample(states, n, replace = TRUE,
                            prob = c(0.25, 0.15, 0.12, 0.10, 0.09,
                                     0.08, 0.07, 0.06, 0.05, 0.03)),
  store_id        = paste0("STR-", sample(sprintf("%03d", 1:40), n, replace = TRUE)),
  category        = sample(categories, n, replace = TRUE,
                            prob = c(0.20, 0.22, 0.18, 0.15, 0.13, 0.12)),
  units_sold      = sample(1:50, n, replace = TRUE),
  unit_price_ngn  = round(runif(n, 500, 150000), -1),
  discount_pct    = sample(c(0, 0, 0, 5, 10, 15, 20), n, replace = TRUE),
  payment_method  = sample(payment_methods, n, replace = TRUE,
                            prob = c(0.30, 0.35, 0.20, 0.15)),
  customer_age    = sample(18:65, n, replace = TRUE),
  customer_gender = sample(c("Male", "Female"), n, replace = TRUE, prob = c(0.54, 0.46)),
  returns         = sample(c(0, 1), n, replace = TRUE, prob = c(0.92, 0.08))
) |>
  # Inject realistic missing values
  mutate(
    customer_age    = ifelse(runif(n) < 0.04, NA, customer_age),
    discount_pct    = ifelse(runif(n) < 0.02, NA, discount_pct),
    customer_gender = ifelse(runif(n) < 0.01, NA, customer_gender)
  )

cat("Dataset created:", nrow(raw_retail), "rows x", ncol(raw_retail), "columns\n")

Dataset created: 1500 rows x 12 columns


## Cell 3: First Look — glimpse()

### Explanation

`glimpse()` is the tidyverse alternative to `str()`. For each column it shows the column name, the R data type, and the first few values, all in a compact single-row-per-column layout.

**Data types to verify:**
- `<date>` for the `date` column — if it reads as `<chr>`, `as.Date()` conversion is needed
- `<dbl>` for numeric columns like `unit_price_ngn` and `revenue_ngn`
- `<chr>` for character columns like `state`, `category`, `payment_method`
- `<int>` for integer columns like `units_sold`, `returns`

If a numeric column loaded as `<chr>`, it usually means non-numeric characters (commas in "1,500" or currency symbols like "₦") were present in the raw data. Fix with `readr::parse_number()` which strips those characters automatically.


In [3]:
# glimpse() shows column names, data types, and first few values
glimpse(raw_retail)

Rows: 1,500
Columns: 12
$ transaction_id  <chr> "TXN-00001", "TXN-00002", "TXN-00003", "TXN-00004", "T…
$ date            <date> 2023-02-18, 2023-11-17, 2023-06-02, 2023-03-15, 2023-…
$ state           <chr> "Oyo", "Kano", "Delta", "Ondo", "Oyo", "Ondo", "Enugu"…
$ store_id        <chr> "STR-032", "STR-015", "STR-032", "STR-012", "STR-017",…
$ category        <chr> "Health & Beauty", "Fashion & Apparel", "Fashion & App…
$ units_sold      <int> 20, 47, 30, 15, 1, 8, 41, 50, 47, 40, 13, 46, 14, 42, …
$ unit_price_ngn  <dbl> 59330, 22380, 42950, 123460, 139030, 97820, 3370, 3253…
$ discount_pct    <dbl> 10, 10, 15, 0, 20, 0, 0, 0, 15, 15, 15, 0, 5, 5, 0, 0,…
$ payment_method  <chr> "Bank Transfer", "Cash", "Cash", "POS", "Cash", "POS",…
$ customer_age    <int> 27, 29, 30, NA, 57, 46, 39, 18, 58, 54, 26, 62, 24, 34…
$ customer_gender <chr> "Male", "Female", "Female", "Male", "Male", "Male", "M…
$ returns         <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, …


## Cell 4: View First 10 Rows — head()

### Explanation

`head(df, n)` returns the first `n` rows. This is the fastest visual sanity check:
- Are transaction IDs formatted correctly? (TXN-00001, not TXN-1)
- Are dates in YYYY-MM-DD format?
- Are prices in a realistic Naira range (₦500–₦150,000)?
- Are category labels spelled consistently? (no mixed case like 'electronics' vs 'Electronics')

In a Jupyter notebook with the IRkernel, tibbles print as clean interactive-style tables showing column types below the header row.


In [4]:
head(raw_retail, 10)

transaction_id,date,state,store_id,category,units_sold,unit_price_ngn,discount_pct,payment_method,customer_age,customer_gender,returns
<chr>,<date>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<chr>,<int>,<chr>,<dbl>
TXN-00001,2023-02-18,Oyo,STR-032,Health & Beauty,20,59330,10,Bank Transfer,27,Male,0
TXN-00002,2023-11-17,Kano,STR-015,Fashion & Apparel,47,22380,10,Cash,29,Female,0
TXN-00003,2023-06-02,Delta,STR-032,Fashion & Apparel,30,42950,15,Cash,30,Female,0
TXN-00004,2023-03-15,Ondo,STR-012,Electronics,15,123460,0,POS,NA,Male,0
TXN-00005,2023-08-16,Oyo,STR-017,Agriculture & Farming,1,139030,20,Cash,57,Male,0
TXN-00006,2023-05-26,Ondo,STR-034,Agriculture & Farming,8,97820,0,POS,46,Male,0
TXN-00007,2023-05-02,Enugu,STR-007,Fashion & Apparel,41,3370,0,POS,39,Male,0
TXN-00008,2023-02-18,Delta,STR-008,Electronics,50,32530,0,POS,18,Female,0
TXN-00009,2023-05-08,Lagos,STR-016,Health & Beauty,47,148760,15,Bank Transfer,58,Female,0


## Cell 5: Rich Summary Statistics — skim()

### Explanation

`skim()` from the `skimr` package is one of the most useful EDA functions in R. It divides output into sections by data type and provides far more than `summary()`:

**For numeric columns:**
- `n_missing` — count of NA values
- `complete_rate` — proportion of non-missing values (1.0 = no missing)
- `mean`, `sd` — central tendency and spread
- `p0`, `p25`, `p50`, `p75`, `p100` — full percentile profile
- `hist` — inline Unicode histogram showing the distribution shape

**For character columns:**
- `n_unique` — number of distinct values (e.g., 10 unique states)
- `min_len`, `max_len` — shortest and longest string lengths

**For date columns:**
- `min`, `max` — earliest and latest dates

The inline histograms are especially valuable: a right-skewed histogram for `unit_price_ngn` immediately shows most transactions are low-priced with a few expensive outliers — information that would require several extra lines to detect otherwise.


In [5]:
skim(raw_retail)

,skim_type,skim_variable,n_missing,complete_rate,Date.min,Date.max,Date.median,Date.n_unique,character.min,character.max,⋯,character.n_unique,character.whitespace,numeric.mean,numeric.sd,numeric.p0,numeric.p25,numeric.p50,numeric.p75,numeric.p100,numeric.hist
,<chr>,<chr>,<int>,<dbl>,<date>,<date>,<date>,<int>,<int>,<int>,⋯,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,Date,date,0,1.0000000,2023-01-01,2023-12-31,2023-06-29,359,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2,character,transaction_id,0,1.0000000,NA,NA,NA,NA,9,9,⋯,1500,0,NA,NA,NA,NA,NA,NA,NA,NA
3,character,state,0,1.0000000,NA,NA,NA,NA,3,7,⋯,10,0,NA,NA,NA,NA,NA,NA,NA,NA
4,character,store_id,0,1.0000000,NA,NA,NA,NA,7,7,⋯,40,0,NA,NA,NA,NA,NA,NA,NA,NA
5,character,category,0,1.0000000,NA,NA,NA,NA,11,21,⋯,6,0,NA,NA,NA,NA,NA,NA,NA,NA
6,character,payment_method,0,1.0000000,NA,NA,NA,NA,3,13,⋯,4,0,NA,NA,NA,NA,NA,NA,NA,NA
7,character,customer_gender,13,0.9913333,NA,NA,NA,NA,4,6,⋯,2,0,NA,NA,NA,NA,NA,NA,NA,NA
8,numeric,units_sold,0,1.0000000,NA,NA,NA,NA,NA,NA,⋯,NA,NA,2.631467e+01,1.428938e+01,1,15.0,27,38.25,50,▆▇▇▇▇
9,numeric,unit_price_ngn,0,1.0000000,NA,NA,NA,NA,NA,NA,⋯,NA,NA,7.609239e+04,4.384790e+04,560,37687.5,78200,113977.50,149990,▇▇▇▇▇


## Cell 6: Missing Value Analysis

### Explanation

The goal is a clean ranked summary: which columns have missing values, how many, and what percentage they represent. Three key techniques:

**`summarise(across(everything(), ~ sum(is.na(.))))`**
- `across(columns, function)` applies the same function to multiple columns simultaneously
- `everything()` selects all columns
- `~ sum(is.na(.))` is a lambda (anonymous) function: `~` introduces it, `.` is the current column
- This replaces writing `sum(is.na(col1)), sum(is.na(col2)), ...` for every column individually

**`pivot_longer(everything(), names_to='column', values_to='missing_count')`**
- `summarise(across(...))` produces ONE row with one column per variable (wide format)
- `pivot_longer()` reshapes it to long format: one row per variable, two columns (name + count)
- Long format is easier to filter, sort, and display in a table

**`arrange(desc(missing_count))`**
- Sorts so columns with the most missing values appear first
- `desc()` reverses the default ascending sort

**Decision rule for action:** Industry convention is that columns with >5% missing values need deliberate handling (imputation or removal). Columns below 5% can usually be safely imputed with median (numeric) or mode/explicit category (character).


In [6]:
missing_summary <- raw_retail |>
  summarise(across(everything(), ~ sum(is.na(.)))) |>
  pivot_longer(everything(),
               names_to  = "column",
               values_to = "missing_count") |>
  mutate(
    missing_pct = round(missing_count / nrow(raw_retail) * 100, 2),
    has_missing = missing_count > 0
  ) |>
  arrange(desc(missing_count))

# Display as a clean table
missing_summary |> kable(caption = "Missing Value Summary by Column")



Table: Missing Value Summary by Column

|column          | missing_count| missing_pct|has_missing |
|:---------------|-------------:|-----------:|:-----------|
|customer_age    |            63|        4.20|TRUE        |
|discount_pct    |            36|        2.40|TRUE        |
|customer_gender |            13|        0.87|TRUE        |
|transaction_id  |             0|        0.00|FALSE       |
|date            |             0|        0.00|FALSE       |
|state           |             0|        0.00|FALSE       |
|store_id        |             0|        0.00|FALSE       |
|category        |             0|        0.00|FALSE       |
|units_sold      |             0|        0.00|FALSE       |
|unit_price_ngn  |             0|        0.00|FALSE       |
|payment_method  |             0|        0.00|FALSE       |
|returns         |             0|        0.00|FALSE       |

## Cell 7: Clean Column Names — janitor

### Explanation

`clean_names()` from the `janitor` package applies a consistent naming convention to all columns:
- Convert to lowercase
- Replace spaces, dots, and special characters with underscores
- Remove leading and trailing underscores
- Handle duplicate names by appending numeric suffixes

**Why this matters:** Column names with spaces must be referenced with backticks in R: `` `Customer Age` ``. After `clean_names()`, they become plain `customer_age` — no backticks needed, no typos possible.

**Best practice:** Always call `clean_names()` immediately after loading any external dataset, before writing any analysis code. If you write 20 functions referencing `'Customer Age'` and then clean the names, all 20 references break. Clean first, analyse second.

Our dataset was built with clean names already, but this step is included to establish the habit and demonstrate the function for when you work with real external data.


In [7]:
# Apply snake_case naming convention to all columns
retail <- raw_retail |>
  clean_names()

# Verify column names
cat("Column names after clean_names():\n")
print(names(retail))

Column names after clean_names():
 [1] "transaction_id"  "date"            "state"           "store_id"       
 [5] "category"        "units_sold"      "unit_price_ngn"  "discount_pct"   
 [9] "payment_method"  "customer_age"    "customer_gender" "returns"        


## Cell 8: Feature Engineering — Derive New Columns

### Explanation

Feature engineering creates new analytical columns from existing data. The variables you engineer often matter more than the modelling algorithm you choose — this is consistently one of the highest-value activities in any data project.

**`left_join(state_region, by = 'state')`**
Merges the region lookup table onto the main dataset. A left join keeps ALL rows from the left table (`retail`) and brings in matching columns from the right (`state_region`). The `by` argument names the key column(s). If a row in `retail` has no match in `state_region`, joined columns will be `NA`.

**Revenue with NA-safe calculation:**
```r
revenue_ngn = units_sold * unit_price_ngn * (1 - discount_pct / 100)
revenue_ngn = replace_na(revenue_ngn, units_sold * unit_price_ngn)
```
When `discount_pct` is `NA`, R propagates `NA` through arithmetic (any number × NA = NA). The second line catches these NAs and substitutes the undiscounted revenue. `replace_na(x, value)` replaces every NA in `x` with `value`.

**`case_when()` — multi-condition categorisation:**
R's vectorised alternative to nested `ifelse()`. Conditions are evaluated top-to-bottom; the first match wins. The final `TRUE ~` is the catch-all default. Equivalent to Excel's nested IF: `=IF(A1<5000,"Budget",IF(A1<30000,"Mid-Range",IF(A1<80000,"Premium","Luxury")))`

**lubridate date extraction:**

| Function | Returns | Key argument |
|----------|---------|-------------|
| `month(date, label=TRUE, abbr=TRUE)` | Ordered factor: Jan, Feb... | `label=TRUE` gives names not numbers |
| `quarter(date)` | Integer 1–4 | Paste 'Q' prefix to get Q1, Q2... |
| `week(date)` | Integer 1–53 | ISO week number |
| `wday(date, label=TRUE, abbr=FALSE)` | Ordered factor: Monday... | Sorted Mon–Sun on plot axes |

**`%in%` operator:** `day_of_week %in% c('Saturday', 'Sunday')` returns TRUE/FALSE for each row — cleaner than writing `day_of_week == 'Saturday' | day_of_week == 'Sunday'`.


In [8]:
retail <- retail |>
  # Join geopolitical region from lookup table
  left_join(state_region, by = "state") |>
  mutate(
    # Revenue after discount (NA-safe: replace_na handles missing discount_pct)
    revenue_ngn      = units_sold * unit_price_ngn * (1 - discount_pct / 100),
    revenue_ngn      = replace_na(revenue_ngn, units_sold * unit_price_ngn),

    # Date decomposition with lubridate
    month            = month(date, label = TRUE, abbr = TRUE),
    quarter          = paste0("Q", quarter(date)),
    week_number      = week(date),
    day_of_week      = wday(date, label = TRUE, abbr = FALSE),
    is_weekend       = day_of_week %in% c("Saturday", "Sunday"),

    # Price tier using case_when() — evaluated top to bottom, first match wins
    price_tier       = case_when(
      unit_price_ngn < 5000  ~ "Budget",
      unit_price_ngn < 30000 ~ "Mid-Range",
      unit_price_ngn < 80000 ~ "Premium",
      TRUE                   ~ "Luxury"
    ),

    # Age group bins
    age_group        = case_when(
      customer_age < 25 ~ "18-24",
      customer_age < 35 ~ "25-34",
      customer_age < 45 ~ "35-44",
      customer_age < 55 ~ "45-54",
      TRUE              ~ "55+"
    ),

    # Discount flag
    discount_applied = !is.na(discount_pct) & discount_pct > 0
  )

cat("Dataset now has", ncol(retail), "columns (was 12)\n")
glimpse(retail)

ERROR: [1m[33mError[39m in `mutate()`:[22m
[1m[22m[36mℹ[39m In argument: `revenue_ngn = replace_na(revenue_ngn, units_sold *
  unit_price_ngn)`.
[1mCaused by error in `replace_na()`:[22m
[1m[22m[33m![39m Replacement for `data` must be length 1, not length 1500.


## Cell 9: Handle Missing Values — Imputation

### Explanation

Three imputation strategies are applied, each chosen for the specific variable:

| Column | Strategy | Justification |
|--------|----------|---------------|
| `customer_age` | **Median imputation** | The median is resistant to extreme values. Ages cluster in the mid-range so median is a reasonable substitute. Mean is pulled by outliers. `na.rm=TRUE` computes median ignoring NAs. |
| `discount_pct` | **Zero imputation** | Domain logic: a missing discount record most plausibly means no discount was applied. Zero is the correct business default, not a statistical estimate. |
| `customer_gender` | **Explicit 'Unknown' category** | Preserves the information that gender was not captured. The proportion of 'Unknown' records is itself analytically meaningful — it reveals data collection gaps. |

**`ifelse(condition, value_if_true, value_if_false)`** works element-wise on vectors:
- `is.na(customer_age)` returns TRUE/FALSE for each row
- Where TRUE, the imputed value is substituted; where FALSE, the original value is kept

After this cell, `sum(is.na(retail))` should return 0 — confirming a completely clean dataset.


In [ ]:
retail <- retail |>
  mutate(
    # Median imputation for continuous variable
    customer_age    = ifelse(is.na(customer_age),
                             median(customer_age, na.rm = TRUE),
                             customer_age),

    # Domain-logic zero imputation for discount
    discount_pct    = ifelse(is.na(discount_pct), 0, discount_pct),

    # Explicit unknown category for gender
    customer_gender = ifelse(is.na(customer_gender), "Unknown", customer_gender)
  )

cat("Missing values remaining after imputation:", sum(is.na(retail)), "\n")

## Cell 10: Overall Revenue Summary Statistics

### Explanation

`summarise()` without a preceding `group_by()` collapses the entire data frame to one row. Each expression becomes one column in the output.

**Key functions inside `summarise()`:**
- `n()` — counts rows in the current group; no column argument needed
- `sum()`, `mean()`, `median()`, `min()`, `max()` — standard aggregation functions
- `round(sum(returns) / n() * 100, 2)` — within `summarise()`, `n()` refers to the group count, so this computes the return rate as a percentage rounded to two decimal places

**`mutate(across(ends_with('_ngn'), ~ comma(.)))`**
Formats all columns ending in `'_ngn'` with comma separators in one expression. Without `across()`, you would write `mutate(col1 = comma(col1), col2 = comma(col2), ...)` for every Naira column individually.


In [ ]:
retail |>
  summarise(
    total_transactions = n(),
    total_revenue_ngn  = sum(revenue_ngn),
    avg_revenue_ngn    = mean(revenue_ngn),
    median_revenue_ngn = median(revenue_ngn),
    min_revenue_ngn    = min(revenue_ngn),
    max_revenue_ngn    = max(revenue_ngn),
    total_units_sold   = sum(units_sold),
    total_returns      = sum(returns),
    return_rate_pct    = round(sum(returns) / n() * 100, 2)
  ) |>
  mutate(across(ends_with("_ngn"), ~ comma(round(., 0)))) |>
  kable(caption = "Overall Revenue Summary — NigeriaRetail Co. 2023")

## Cell 11: Revenue by State

### Explanation

This is the classic `group_by()` + `summarise()` pattern — the most frequently used combination in dplyr. **`group_by(state, region)`** splits the data into groups; every subsequent operation is performed independently within each group. Grouping by both state and region means region is preserved in the output as a grouping variable rather than needing to be joined back later.

**`.groups = 'drop'`** removes the group metadata after `summarise()`. Without this, the output remains a grouped tibble and subsequent sorts or mutates may behave unexpectedly. As of dplyr 1.0, R issues a warning if you do not specify this explicitly.

**`arrange(desc(total_revenue_ngn))`** sorts in descending order. Always sort your group summaries by the metric of interest before displaying — the table is far more scannable when the top performer is at the top.


In [ ]:
revenue_by_state <- retail |>
  group_by(state, region) |>
  summarise(
    transactions      = n(),
    total_revenue_ngn = sum(revenue_ngn),
    avg_revenue_ngn   = mean(revenue_ngn),
    total_units       = sum(units_sold),
    .groups = "drop"
  ) |>
  arrange(desc(total_revenue_ngn))

revenue_by_state |>
  mutate(across(ends_with("_ngn"), ~ paste0("N", comma(round(., 0))))) |>
  kable(caption = "Revenue by Nigerian State (2023)")

## Cell 12: Revenue by Product Category

### Explanation

Same `group_by()` + `summarise()` pattern applied to `category`. Notice the return rate computation: **`sum(returns) / n() * 100`**. Since `returns` is binary (0 or 1), `sum(returns)` counts the number of returned transactions. Dividing by `n()` gives the proportion; multiplying by 100 converts to a percentage. This pattern (`sum(binary_col) / n()`) is very common in retail analytics for computing event rates.

The `starts_with('avg_unit')` selector in `across()` catches the `avg_unit_price` column alongside the `_ngn` columns, applying consistent Naira formatting to both.


In [ ]:
revenue_by_cat <- retail |>
  group_by(category) |>
  summarise(
    transactions      = n(),
    total_revenue_ngn = sum(revenue_ngn),
    avg_unit_price    = mean(unit_price_ngn),
    avg_units_sold    = round(mean(units_sold), 2),
    return_rate_pct   = round(sum(returns) / n() * 100, 2),
    .groups = "drop"
  ) |>
  arrange(desc(total_revenue_ngn))

revenue_by_cat |>
  mutate(across(ends_with("_ngn") | starts_with("avg_unit"),
                ~ paste0("N", comma(round(., 0))))) |>
  kable(caption = "Revenue by Product Category (2023)")

## Cell 13: Monthly Revenue Trend

### Explanation

Grouping by `month` produces an **ordered factor** (Jan through Dec) because we used `label = TRUE` in the `month()` call during feature engineering. R respects factor ordering when displaying tables and plotting — so January automatically comes before February without any manual sorting. This is one of the key advantages of using `label = TRUE` in lubridate: you get correct chronological ordering 'for free'.

Grouping by both `month` AND `quarter` allows the `quarter` column to appear in the output for context, even though we are not aggregating by quarter here. This is a common technique: include extra descriptive grouping variables that you want preserved in the output.


In [ ]:
monthly_revenue <- retail |>
  group_by(month, quarter) |>
  summarise(
    total_revenue_ngn = sum(revenue_ngn),
    transactions      = n(),
    .groups = "drop"
  )

monthly_revenue |>
  mutate(total_revenue_ngn = paste0("N", comma(round(total_revenue_ngn, 0)))) |>
  kable(caption = "Monthly Revenue Summary — NigeriaRetail Co. 2023")

## Cell 14: Payment Method Analysis

### Explanation

**`share_pct = round(n() / nrow(retail) * 100, 2)`** — inside `summarise()`, `n()` gives the count in the current group, while `nrow(retail)` gives the total rows in the full data frame. This ratio gives each payment method's share of all transactions as a percentage. This pattern — `n() / nrow(df)` — is the standard way to compute group proportions in dplyr when you need the denominator to be the full dataset rather than just the group.

**Business context:** POS dominance reflects the rapid expansion of card infrastructure in Nigerian retail following CBN cashless policy initiatives. Mobile Money at ~15% represents a growing channel that warrants continued investment in USSD and wallet integrations.


In [ ]:
retail |>
  group_by(payment_method) |>
  summarise(
    count         = n(),
    share_pct     = round(n() / nrow(retail) * 100, 2),
    avg_revenue   = round(mean(revenue_ngn), 0),
    total_revenue = round(sum(revenue_ngn), 0),
    .groups = "drop"
  ) |>
  arrange(desc(count)) |>
  kable(caption = "Payment Method Distribution and Revenue Contribution")

## Cell 15: Customer Demographics Analysis

### Explanation

**`filter(customer_gender != 'Unknown')`** removes the imputed 'Unknown' records before computing gender-based statistics. Including 'Unknown' as a gender category would distort the gender comparison and mislead the reader.

Grouping by two variables (`customer_gender` and `age_group`) produces a cross-tabulation — every combination of gender and age group appears in the output. **`arrange(customer_gender, age_group)`** sorts alphabetically by gender first, then by age group within each gender, producing a readable grid structure.

**Business insight:** The 25-34 cohort typically generates the highest revenue across both genders in Nigerian retail — this is the digitally active, income-earning millennial segment that should anchor marketing investment.


In [ ]:
retail |>
  filter(customer_gender != "Unknown") |>
  group_by(customer_gender, age_group) |>
  summarise(
    count         = n(),
    avg_revenue   = round(mean(revenue_ngn), 0),
    total_revenue = round(sum(revenue_ngn), 0),
    .groups = "drop"
  ) |>
  arrange(customer_gender, age_group) |>
  kable(caption = "Revenue by Customer Gender and Age Group")

## Cell 16: Discount Impact Analysis

### Explanation

`discount_applied` is a logical column (TRUE/FALSE). Grouping by both `discount_applied` and `discount_pct` shows the breakdown by exact discount level.

**`filter(!(discount_applied == FALSE & discount_pct > 0))`** removes an impossible combination — rows where `discount_applied` is FALSE but `discount_pct` is greater than 0 (an artefact of our NA handling). The `!()` negates the condition inside.

**Business question:** Does offering a larger discount actually increase units sold? If `avg_units_sold` is similar across discount levels, discounts are reducing margin without driving volume — a common finding in Nigerian FMCG retail. Also: do discounted transactions have a higher return rate? Higher returns on discounted items could indicate impulse purchasing that does not match customer needs.


In [ ]:
retail |>
  group_by(discount_applied, discount_pct) |>
  summarise(
    transactions   = n(),
    avg_units_sold = round(mean(units_sold), 2),
    avg_revenue    = round(mean(revenue_ngn), 0),
    return_rate    = round(mean(returns) * 100, 2),
    .groups = "drop"
  ) |>
  filter(!(discount_applied == FALSE & discount_pct > 0)) |>
  kable(caption = "Discount Level vs Revenue, Units Sold, and Return Rate")

## Cell 17: Weekend vs Weekday Performance

### Explanation

`is_weekend` is a logical column. After summarising, **`mutate(day_type = ifelse(is_weekend, 'Weekend', 'Weekday'))`** converts the TRUE/FALSE to a readable label. **`select(day_type, ...)`** then reorders and picks only the columns needed for the display table, dropping `is_weekend` since it has been replaced by the more readable label.

**Operational implication:** If weekend average revenue is higher per transaction, it suggests customers make larger planned purchases on weekends. If weekday transaction volume is lower, there may be staffing or promotional opportunities on weekdays to smooth the revenue curve.


In [ ]:
retail |>
  group_by(is_weekend) |>
  summarise(
    transactions    = n(),
    avg_revenue_ngn = round(mean(revenue_ngn), 0),
    total_revenue   = round(sum(revenue_ngn), 0),
    avg_units       = round(mean(units_sold), 2),
    .groups = "drop"
  ) |>
  mutate(day_type = ifelse(is_weekend, "Weekend", "Weekday")) |>
  select(day_type, transactions, avg_revenue_ngn, total_revenue, avg_units) |>
  kable(caption = "Weekend vs Weekday Sales Performance")

## Cell 18: Price Tier Cross-Tabulation — pivot_wider()

### Explanation

**`count(price_tier, category)`** is a shortcut for `group_by(price_tier, category) |> summarise(n = n())`. It counts transactions for every combination of the two variables, producing a long-format table with three columns: `price_tier`, `category`, and `n`.

**`pivot_wider(names_from = price_tier, values_from = n, values_fill = 0)`**
- `names_from = price_tier` — each unique value in `price_tier` becomes a new column
- `values_from = n` — the transaction counts fill the cells
- `values_fill = 0` — combinations that do not exist get 0 instead of NA

This is R's equivalent of an Excel PivotTable. The `pivot_wider()` / `pivot_longer()` pair from `tidyr` handles all reshape operations. The rule: use `pivot_longer()` to go from wide to long (needed for ggplot2), and `pivot_wider()` to go from long to wide (needed for display tables and cross-tabulations).


In [ ]:
retail |>
  count(price_tier, category) |>
  pivot_wider(
    names_from   = price_tier,
    values_from  = n,
    values_fill  = 0
  ) |>
  kable(caption = "Transaction Count by Category and Price Tier")

## Cell 19: Chart 1 — Total Revenue by State (Horizontal Bar Chart)

### ggplot2 Grammar Explanation

Every ggplot2 chart follows the same grammar:
1. `ggplot(data, aes(...))` — defines the dataset and aesthetic mappings
2. `geom_*()` — adds a geometric layer (the visual mark)
3. `scale_*()` — controls how data values map to visual properties
4. `labs()` — sets all text labels
5. `theme_*()` and `theme()` — controls non-data appearance

**`fct_reorder(state, total_revenue)`** reorders the `state` factor levels by revenue. Without this, bars sort alphabetically. With `coord_flip()`, the reordered factor reads top-to-bottom from highest to lowest revenue.

**`coord_flip()`** rotates the chart 90 degrees — horizontal bars are easier to read when category labels are long, as they are with Nigerian state names.

**`scale_fill_gradient(low=..., high=...)`** applies a continuous colour gradient — both bar length AND colour encode the same revenue value. This double-encoding makes the chart more scannable: high-revenue states are both longer and darker.

**`scale_y_continuous(labels = function(x) paste0('N', x, 'M'))`** — an anonymous function applied to every axis tick label, formatting them as N5M, N10M etc.

**`theme(panel.grid.major.y = element_blank())`** removes horizontal grid lines, which are redundant when bars themselves provide the visual reference.


In [ ]:
retail |>
  group_by(state) |>
  summarise(total_revenue = sum(revenue_ngn), .groups = "drop") |>
  mutate(state = fct_reorder(state, total_revenue)) |>
  ggplot(aes(x = state, y = total_revenue / 1e6, fill = total_revenue)) +
  geom_col(show.legend = FALSE) +
  coord_flip() +
  scale_fill_gradient(low = "#8ecae6", high = "#023047") +
  scale_y_continuous(labels = function(x) paste0("N", x, "M")) +
  labs(
    title    = "Total Revenue by State (2023)",
    subtitle = "NigeriaRetail Co. | 1,500 Transactions",
    x        = NULL,
    y        = "Total Revenue (Millions NGN)",
    caption  = "Source: NigeriaRetail Co. Dataset | Prodigy Training Hub"
  ) +
  theme_minimal(base_size = 13) +
  theme(
    plot.title         = element_text(face = "bold"),
    plot.subtitle      = element_text(colour = "grey50"),
    panel.grid.major.y = element_blank()
  )

## Cell 20: Chart 2 — Revenue by Category (Lollipop Chart)

### Explanation

A lollipop chart replaces solid bars with a stick (`geom_segment()`) and a dot (`geom_point()`). It conveys identical information with significantly less ink — particularly effective when comparing many ranked categories.

**Two-layer construction:**
- `geom_segment(aes(xend = category, yend = 0))` draws the vertical stick from each category's revenue value down to the zero baseline. `xend = category` keeps the line at the same x position as the point above.
- `geom_point(size = 5)` draws the dot. It must come AFTER `geom_segment()` in the code so it renders on top of (not behind) the stick. In ggplot2, layers render in source order.

**When to use lollipop vs bar:** Use a lollipop when (1) you have 6+ categories, (2) the differences between values are the story rather than the magnitudes themselves, or (3) you want a lighter visual aesthetic.


In [ ]:
retail |>
  group_by(category) |>
  summarise(total_revenue = sum(revenue_ngn), .groups = "drop") |>
  mutate(category = fct_reorder(category, total_revenue)) |>
  ggplot(aes(x = category, y = total_revenue / 1e6)) +
  geom_segment(aes(xend = category, yend = 0), colour = "#457b9d", linewidth = 1) +
  geom_point(size = 5, colour = "#e63946") +
  coord_flip() +
  scale_y_continuous(labels = function(x) paste0("N", x, "M")) +
  labs(
    title   = "Revenue by Product Category (2023)",
    x       = NULL,
    y       = "Total Revenue (Millions NGN)",
    caption = "Source: NigeriaRetail Co. Dataset | Prodigy Training Hub"
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold"),
        panel.grid.major.y = element_blank())

## Cell 21: Chart 3 — Monthly Revenue Trend (Line Chart)

### Explanation

**`as.integer(month)`** converts the ordered factor (Jan=1, Feb=2...) to integers so that `scale_x_continuous()` can control tick placement and labels precisely.

**`group = 1`** inside `aes()` is critical for line charts. Without it, ggplot2 tries to draw a separate line for each x value, producing disconnected dots. `group = 1` tells ggplot2 to connect all 12 monthly points into one continuous line.

**Two-layer time series:** `geom_line()` shows the trend; `geom_point(aes(colour = quarter))` adds coloured dots that communicate quarterly position. The colour aesthetic is set only on the points layer (not on the line), so the line stays green while points are quarter-coloured.

**`scale_colour_manual(values = c(Q1=..., Q2=..., Q3=..., Q4=...))`** — named vectors guarantee the right colour maps to the right label regardless of data ordering.


In [ ]:
monthly_revenue |>
  ggplot(aes(x = as.integer(month), y = total_revenue_ngn / 1e6, group = 1)) +
  geom_line(colour = "#2a9d8f", linewidth = 1.2) +
  geom_point(aes(colour = quarter), size = 4) +
  scale_x_continuous(breaks = 1:12, labels = month.abb) +
  scale_y_continuous(labels = function(x) paste0("N", x, "M")) +
  scale_colour_manual(values = c(Q1 = "#264653", Q2 = "#2a9d8f",
                                  Q3 = "#e9c46a", Q4 = "#e76f51")) +
  labs(
    title   = "Monthly Revenue Trend (2023)",
    x       = NULL,
    y       = "Revenue (Millions NGN)",
    colour  = "Quarter",
    caption = "Source: NigeriaRetail Co. Dataset | Prodigy Training Hub"
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold"))

## Cell 22: Chart 4 — Payment Method Distribution (Donut Chart)

### Explanation

ggplot2 has no native donut geometry. The standard technique has three steps:
1. Build a stacked bar with `geom_col()`, mapping `y` to percentage and `fill` to category
2. Apply `coord_polar(theta = 'y')` to wrap the y-axis into a circle
3. Limit x with `xlim(0.5, 2.5)` — the hole appears because x starts at 0.5 (not 0)

**`position_stack(vjust = 0.5)`** in `geom_text()` places each label at the vertical midpoint of its slice. Without it, all labels stack at the top edge of each segment.

**`theme_void()`** removes all axes, grid lines, and background — appropriate for circular charts where Cartesian axes are meaningless.

**When to use a donut:** Part-to-whole relationships with 3–5 categories. For 6+ categories, a sorted horizontal bar chart is always clearer.


In [ ]:
payment_data <- retail |>
  count(payment_method) |>
  mutate(
    pct   = n / sum(n),
    label = paste0(payment_method, "\n", percent(pct, accuracy = 0.1))
  )

ggplot(payment_data, aes(x = 2, y = pct, fill = payment_method)) +
  geom_col(width = 1, colour = "white", linewidth = 0.5) +
  coord_polar(theta = "y") +
  xlim(0.5, 2.5) +
  scale_fill_manual(values = c(
    "Cash"          = "#264653",
    "POS"           = "#2a9d8f",
    "Bank Transfer" = "#e9c46a",
    "Mobile Money"  = "#e76f51"
  )) +
  geom_text(aes(label = label),
            position  = position_stack(vjust = 0.5),
            colour    = "white", size = 3.5, fontface = "bold") +
  labs(
    title   = "Payment Method Distribution (2023)",
    fill    = NULL,
    caption = "Source: NigeriaRetail Co. Dataset | Prodigy Training Hub"
  ) +
  theme_void(base_size = 13) +
  theme(plot.title      = element_text(face = "bold", hjust = 0.5),
        legend.position = "none")

## Cell 23: Chart 5 — Revenue by Region and Category (Faceted Bar Chart)

### Explanation

**`facet_wrap(~ region, scales = 'free_y')`** creates a separate panel for each region. The `~` is R's formula notation meaning 'facet by the variable on the right'. `scales = 'free_y'` gives each panel its own y-axis scale — appropriate here because South West (Lagos-dominated) revenue would dwarf North East on a fixed scale, making smaller regions unreadable. Use `scales = 'fixed'` only when direct magnitude comparison across panels is the analytical goal.

**`str_wrap(category, width = 12)`** inserts newlines every 12 characters to prevent x-axis label overlap in the small facet panels. `str_wrap()` from `stringr` respects word boundaries — it does not cut words mid-character.

**`scale_fill_brewer(palette = 'Set2')`** uses a ColorBrewer qualitative palette — professionally designed colours for categorical data that remain distinguishable for colour-blind viewers and in greyscale print.


In [ ]:
retail |>
  group_by(region, category) |>
  summarise(total_revenue = sum(revenue_ngn) / 1e6, .groups = "drop") |>
  mutate(category = str_wrap(category, width = 12)) |>
  ggplot(aes(x = category, y = total_revenue, fill = region)) +
  geom_col(position = "dodge") +
  facet_wrap(~ region, scales = "free_y") +
  scale_fill_brewer(palette = "Set2") +
  scale_y_continuous(labels = function(x) paste0("N", x, "M")) +
  labs(
    title   = "Revenue by Category across Geopolitical Regions (2023)",
    x       = NULL,
    y       = "Revenue (Millions NGN)",
    fill    = "Region",
    caption = "Source: NigeriaRetail Co. Dataset | Prodigy Training Hub"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    plot.title      = element_text(face = "bold"),
    legend.position = "none",
    axis.text.x     = element_text(angle = 35, hjust = 1, size = 8)
  )

## Cell 24: Chart 6 — Unit Price Distribution by Category (Boxplot)

### Explanation

A boxplot communicates five statistics simultaneously:
- **Box:** spans Q1 (25th percentile) to Q3 (75th percentile) — the Interquartile Range (IQR)
- **Line inside box:** median (50th percentile)
- **Whiskers:** extend to the most extreme value within 1.5 × IQR from the box edges
- **Dots beyond whiskers:** individual outliers

Boxplots reveal distribution shape that summary statistics miss. Two categories with the same mean and same price range can have completely different distributions — one might be uniformly spread while the other clusters at extremes. The boxplot shows this immediately.

**`fct_reorder(category, unit_price_ngn, median)`** orders categories by their median price, making the chart read cheapest-to-most-expensive. The third argument specifies the summary function for ordering (default is `mean`; here we use `median` to match the boxplot's central line).

**`outlier.alpha = 0.4`** makes outlier dots semi-transparent to reduce visual clutter when many outliers stack at the same price point.


In [ ]:
retail |>
  mutate(category = fct_reorder(category, unit_price_ngn, median)) |>
  ggplot(aes(x = category, y = unit_price_ngn / 1000, fill = category)) +
  geom_boxplot(alpha = 0.7, outlier.colour = "#e63946", outlier.alpha = 0.4) +
  coord_flip() +
  scale_fill_brewer(palette = "Pastel1") +
  scale_y_continuous(labels = function(x) paste0("N", x, "K")) +
  labs(
    title   = "Unit Price Distribution by Product Category",
    x       = NULL,
    y       = "Unit Price (Thousands NGN)",
    caption = "Source: NigeriaRetail Co. Dataset | Prodigy Training Hub"
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold"), legend.position = "none")

## Cell 25: Chart 7 — Revenue Heatmap (Month × Day of Week)

### Explanation

A heatmap encodes a numeric value as colour across a two-dimensional grid. It is ideal for detecting patterns across two categorical dimensions simultaneously — in this case, which month-day combinations generate the highest average transaction values.

**`geom_tile()`** draws a filled rectangle for each row in the data. The fill colour maps to `avg_revenue`. Adding `colour = 'white', linewidth = 0.5` draws white gridlines between tiles, making individual cells easier to distinguish.

**`factor(day_of_week, levels = c('Monday', ...))`** explicitly re-levels the day factor to run Monday through Sunday from top to bottom. Without this, ggplot2 sorts alphabetically (Friday, Monday, Saturday...) — not the expected day-of-week order.

**`scale_fill_gradient(low='#caf0f8', high='#023e8a')`** maps low values to pale blue and high values to dark navy. The choice is intentional: darker = higher revenue is an intuitive mapping that reads without a legend.


In [ ]:
retail |>
  group_by(month, day_of_week) |>
  summarise(avg_revenue = mean(revenue_ngn) / 1000, .groups = "drop") |>
  mutate(day_of_week = factor(day_of_week,
    levels = c("Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"))) |>
  ggplot(aes(x = month, y = day_of_week, fill = avg_revenue)) +
  geom_tile(colour = "white", linewidth = 0.5) +
  scale_fill_gradient(low  = "#caf0f8", high = "#023e8a",
                      labels = function(x) paste0("N", x, "K")) +
  labs(
    title   = "Average Transaction Value Heatmap — Month x Day of Week",
    x       = NULL,
    y       = NULL,
    fill    = "Avg Revenue",
    caption = "Source: NigeriaRetail Co. Dataset | Prodigy Training Hub"
  ) +
  theme_minimal(base_size = 12) +
  theme(plot.title  = element_text(face = "bold"),
        axis.text.x = element_text(angle = 45, hjust = 1))

## Cell 26: Chart 8 — Revenue by Age Group and Gender (Stacked Bar)

### Explanation

**`position = 'stack'`** stacks the male and female revenue bars on top of each other for each age group, showing both the total revenue per group AND the gender split within each group in a single chart. Compare to:
- `position = 'dodge'` — bars side by side (easier to compare gender within age group)
- `position = 'fill'` — bars normalised to 100% height (easier to compare gender proportion)

**`filter(customer_gender != 'Unknown')`** removes the imputed 'Unknown' category before plotting — including it would misrepresent the gender comparison.

**`scale_fill_manual(values = c(Male='#457b9d', Female='#e63946'))`** assigns explicit brand-consistent colours to each gender, ensuring consistent colour use across all charts in this project that reference gender.


In [ ]:
retail |>
  filter(customer_gender != "Unknown") |>
  group_by(age_group, customer_gender) |>
  summarise(total_revenue = sum(revenue_ngn) / 1e6, .groups = "drop") |>
  ggplot(aes(x = age_group, y = total_revenue, fill = customer_gender)) +
  geom_col(position = "stack") +
  scale_fill_manual(values = c(Male = "#457b9d", Female = "#e63946")) +
  scale_y_continuous(labels = function(x) paste0("N", x, "M")) +
  labs(
    title   = "Total Revenue by Age Group and Gender",
    x       = "Age Group",
    y       = "Revenue (Millions NGN)",
    fill    = "Gender",
    caption = "Source: NigeriaRetail Co. Dataset | Prodigy Training Hub"
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold"))

## Cell 27: Chart 9 — Units Sold vs Revenue by Category (Scatter + Regression)

### Explanation

A scatter plot reveals the relationship between two continuous variables. `facet_wrap(~ category)` creates one panel per category, allowing us to see whether the units-sold/revenue relationship differs across product types.

**`sample_n(500)`** draws a random sample of 500 rows before plotting. With 1,500 points across 6 facets (~250 per panel), overplotting is manageable. For larger datasets, sampling to 200–500 points per panel improves readability without losing the pattern.

**`geom_smooth(method = 'lm', se = FALSE, linewidth = 0.8)`** adds a linear regression line to each panel. `method = 'lm'` specifies ordinary least squares. `se = FALSE` removes the grey confidence interval ribbon — include it (`se = TRUE`, the default) when you want to show uncertainty around the trend.

**`scales = 'free_y'`** gives each facet its own y-axis scale, necessary because Electronics revenue ranges much higher than Food & Beverages.

**`alpha = 0.5`** makes points 50% transparent, reducing overplotting where many transactions have identical units-sold values (1, 5, 10...).


In [ ]:
set.seed(42)  # For reproducible sample
retail |>
  sample_n(500) |>
  ggplot(aes(x = units_sold, y = revenue_ngn / 1000, colour = category)) +
  geom_point(alpha = 0.5, size = 2) +
  geom_smooth(method = "lm", se = FALSE, linewidth = 0.8) +
  facet_wrap(~ category, scales = "free_y") +
  scale_colour_brewer(palette = "Dark2") +
  scale_y_continuous(labels = function(x) paste0("N", x, "K")) +
  labs(
    title   = "Units Sold vs Revenue by Product Category",
    x       = "Units Sold",
    y       = "Revenue (Thousands NGN)",
    caption = "Source: NigeriaRetail Co. Dataset | Prodigy Training Hub"
  ) +
  theme_minimal(base_size = 11) +
  theme(plot.title = element_text(face = "bold"), legend.position = "none")

## Cell 28: Chart 10 — Quarterly Revenue by Geopolitical Region (Grouped Bar)

### Explanation

**`position = 'dodge'`** places bars from different groups side by side within each quarter. This is the standard grouped bar chart. Compare to `position = 'stack'` (bars stacked, showing totals) and `position = 'fill'` (proportional stacks, showing regional share of each quarter's revenue).

This chart answers the question: **do all regions follow the same seasonal pattern, or do some regions peak in different quarters?** A region that peaks in Q3 while others peak in Q4 might have a different product mix (e.g., Agriculture & Farming seasonality driven by harvest cycles).

**`scale_fill_brewer(palette = 'Set1')`** uses a high-contrast ColorBrewer palette suitable for 5-6 categories that need to be clearly distinguished when placed adjacent to each other.


In [ ]:
retail |>
  group_by(quarter, region) |>
  summarise(total_revenue = sum(revenue_ngn) / 1e6, .groups = "drop") |>
  ggplot(aes(x = quarter, y = total_revenue, fill = region)) +
  geom_col(position = "dodge") +
  scale_fill_brewer(palette = "Set1") +
  scale_y_continuous(labels = function(x) paste0("N", x, "M")) +
  labs(
    title   = "Quarterly Revenue by Geopolitical Region",
    x       = "Quarter",
    y       = "Revenue (Millions NGN)",
    fill    = "Region",
    caption = "Source: NigeriaRetail Co. Dataset | Prodigy Training Hub"
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold"))

## Cell 29: Top 10 Stores by Revenue

### Explanation

**`slice_max(order_by = total_revenue, n = 10)`** selects the top 10 rows by `total_revenue`. This is equivalent to `arrange(desc(total_revenue)) |> head(10)` but more concise and self-documenting. Use `slice_min()` for the bottom 10.

**Average basket value** (`avg_basket = mean(revenue_ngn)`) measures the average transaction size per store. A store might rank highly on total revenue because of high transaction volume (many transactions) OR high basket size (few but large transactions). Showing both `transactions` and `avg_basket` together reveals which dynamic is driving each store's total revenue.


In [ ]:
retail |>
  group_by(store_id, state) |>
  summarise(
    transactions  = n(),
    total_revenue = sum(revenue_ngn),
    avg_basket    = mean(revenue_ngn),
    .groups = "drop"
  ) |>
  slice_max(order_by = total_revenue, n = 10) |>
  mutate(
    total_revenue = paste0("N", comma(round(total_revenue, 0))),
    avg_basket    = paste0("N", comma(round(avg_basket, 0)))
  ) |>
  kable(caption = "Top 10 Stores by Total Revenue (2023)")

## Cell 30: Return Rate by Region and Category (Cross-Tab)

### Explanation

This cell produces a cross-tabulated return rate matrix: rows are regions, columns are categories, cells are return percentages. The pipeline:
1. `group_by(region, category)` + `summarise(return_rate = mean(returns) * 100)` — computes return rate for every region-category combination (long format)
2. `pivot_wider(names_from = category, values_from = return_rate)` — reshapes to a cross-tab matrix (wide format)

**`mean(returns) * 100`** works because `returns` is binary (0 or 1). The mean of a binary variable equals the proportion of 1s, which when multiplied by 100 gives the percentage return rate. This is a fundamental statistical identity used throughout retail analytics.

**What to look for:** A high return rate in a specific region-category cell (e.g., Electronics in North West) could indicate counterfeit product infiltration, logistics damage, or poor customer fit. This cross-tab directs the operations team to investigate specific combinations rather than the overall average.


In [ ]:
retail |>
  group_by(region, category) |>
  summarise(
    return_rate = round(mean(returns) * 100, 2),
    .groups = "drop"
  ) |>
  pivot_wider(
    names_from  = category,
    values_from = return_rate,
    values_fill = 0
  ) |>
  kable(caption = "Return Rate (%) by Region and Product Category")

## Cell 31: Correlation Matrix — Numeric Variables

### Explanation

A correlation matrix shows the pairwise linear relationship between all numeric variables. Values range from -1 (perfect negative correlation) to +1 (perfect positive correlation). 0 means no linear relationship.

**`select(units_sold, unit_price_ngn, discount_pct, revenue_ngn, customer_age)`** — we select only the numeric columns meaningful for correlation analysis. Including binary or ID columns would distort the matrix.

**`cor(use = 'complete.obs')`** computes pairwise correlations, excluding rows with any NA value. After our imputation in Cell 9 there should be no NAs, but `use = 'complete.obs'` is a safe default.

**Expected pattern:** `revenue_ngn` should show strong positive correlation with `units_sold` (more units = more revenue) and `unit_price_ngn` (higher price = higher revenue). Weak or negative correlation between `discount_pct` and `revenue_ngn` would confirm that discounts are not lifting overall revenue — supporting the insight from Cell 16.


In [ ]:
retail |>
  select(units_sold, unit_price_ngn, discount_pct, revenue_ngn, customer_age) |>
  cor(use = "complete.obs") |>
  round(3) |>
  kable(caption = "Pairwise Correlation Matrix — Numeric Variables")

## Cell 32: Key Business Insights Summary

### Explanation

The final cell extracts headline figures programmatically and formats them as a printed summary — the kind of output you would present to a non-technical business stakeholder.

**`slice_max(rev, n=1) |> pull(state)`** — `slice_max()` selects the top row by `rev`; `pull()` extracts the `state` column as a plain character vector (not a tibble). This is the idiomatic tidyverse way to extract a single value.

**`cat()`** prints to the console without the `[1]` prefix that `print()` adds. `comma()` from `scales` formats large numbers with thousand separators. Together they produce clean, human-readable output.

---

### Business Insights Interpretation

| Insight | Finding | Recommended Action |
|---------|---------|--------------------|
| Geographic concentration | Lagos dominates revenue (~25% of transactions) | Investigate whether lower-revenue states reflect low demand or insufficient store presence |
| Category performance | Fashion leads in transactions; Electronics leads in per-unit revenue | Balance assortment strategy: volume categories vs. high-margin categories |
| Payment channel | POS leads at ~35%; Mobile Money at ~15% | Invest in POS terminal uptime and USSD/wallet integrations for Mobile Money growth |
| Discount effectiveness | Discount level does not significantly increase units sold | Review discount strategy — if discounts are not driving volume, they are reducing margin without benefit |
| Demographics | 25-34 cohort drives highest revenue | Focus digital marketing investment on this millennial segment via social media and mobile apps |
| Return rate | Overall ~8% return rate | Cross-tabulate by region x category (Cell 30) to find specific problem combinations requiring investigation |

---

### Next Steps

Having completed the EDA, the natural next steps for this dataset are:

1. **Project 2 (Bank Customer Data Wrangling):** Apply dplyr string operations with `stringr`, multi-table joins, and date arithmetic
2. **Statistical Modelling (Tier 2):** Build a linear regression model predicting `revenue_ngn` from `units_sold`, `unit_price_ngn`, `discount_pct`, and `category`
3. **Customer Segmentation (Tier 3 ML):** Apply K-means clustering to segment customers by `customer_age`, `avg_basket`, and `preferred_category`
4. **Demand Forecasting (Tier 2 Time Series):** Model monthly revenue using `fable` and decompose into trend, seasonality, and remainder components


In [ ]:
# Extract headline metrics programmatically
top_state    <- retail |> group_by(state) |> summarise(rev = sum(revenue_ngn)) |>
                  slice_max(rev, n=1) |> pull(state)
top_cat      <- retail |> group_by(category) |> summarise(rev = sum(revenue_ngn)) |>
                  slice_max(rev, n=1) |> pull(category)
top_payment  <- retail |> count(payment_method) |> slice_max(n, n=1) |> pull(payment_method)
best_quarter <- retail |> group_by(quarter) |> summarise(rev = sum(revenue_ngn)) |>
                  slice_max(rev, n=1) |> pull(quarter)
total_rev    <- sum(retail$revenue_ngn)
return_rate  <- mean(retail$returns) * 100
avg_basket   <- mean(retail$revenue_ngn)

cat("\n")
cat("============================================================\n")
cat("       NIGERIARETAIL CO. — EDA KEY INSIGHTS (2023)         \n")
cat("============================================================\n")
cat("  Total Revenue:             N", comma(round(total_rev, 0)), "\n")
cat("  Total Transactions:       ", nrow(retail), "\n")
cat("  Average Basket Size:       N", comma(round(avg_basket, 0)), "\n")
cat("  Top State by Revenue:     ", top_state, "\n")
cat("  Top Product Category:     ", top_cat, "\n")
cat("  Dominant Payment Method:  ", top_payment, "\n")
cat("  Highest Revenue Quarter:  ", best_quarter, "\n")
cat("  Overall Return Rate:      ", round(return_rate, 2), "%\n")
cat("============================================================\n")